In [1]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

modeling_table_path = (
    project_root / "data" / "processed"
    / "household_campaign_modeling.parquet"
)

modeling_table = pd.read_parquet(modeling_table_path)

print(f"Modeling examples: {len(modeling_table):,}")
print(f"Campaigns: {modeling_table['CAMPAIGN'].nunique()}")

Modeling examples: 7,091
Campaigns: 28


In [2]:
campaign_timeline = (
    modeling_table.groupby("CAMPAIGN", as_index=False)
    .agg(
        start_day=("START_DAY", "first"),
        end_day=("END_DAY", "first"),
        examples=("household_key", "size")
    )
    .sort_values(["start_day", "CAMPAIGN"])
)

campaign_timeline.head(14)

,CAMPAIGN,start_day,end_day,examples
23,26,224,264,332
24,27,237,300,12
25,28,259,320,17
26,29,281,334,118
27,30,323,369,361
0,1,346,383,13
1,2,351,383,48
2,3,356,412,12
3,4,372,404,81
4,5,377,411,166


In [3]:
campaign_timeline.iloc[14:]

,CAMPAIGN,start_day,end_day,examples
9,10,463,495,123
10,11,477,523,214
11,12,477,509,170
12,13,504,551,1077
13,14,531,596,224
14,16,561,593,188
15,17,575,607,202
16,18,587,642,1133
17,19,603,635,130
18,20,615,685,244


In [4]:
validation_start_day = 463
test_start_day = 587

train_data = modeling_table.loc[
    modeling_table["END_DAY"] < validation_start_day
].copy()

validation_data = modeling_table.loc[
    (modeling_table["START_DAY"] >= validation_start_day)
    & (modeling_table["END_DAY"] < test_start_day)
].copy()

test_data = modeling_table.loc[
    modeling_table["START_DAY"] >= test_start_day
].copy()

excluded_count = (
    len(modeling_table)
    - len(train_data)
    - len(validation_data)
    - len(test_data)
)

print(f"Training examples: {len(train_data):,}")
print(f"Validation examples: {len(validation_data):,}")
print(f"Test examples: {len(test_data):,}")
print(f"Examples excluded at boundaries: {excluded_count:,}")

Training examples: 2,499
Validation examples: 1,584
Test examples: 2,218
Examples excluded at boundaries: 790


In [5]:
train_campaigns = set(train_data["CAMPAIGN"])
validation_campaigns = set(validation_data["CAMPAIGN"])
test_campaigns = set(test_data["CAMPAIGN"])

assert train_campaigns.isdisjoint(validation_campaigns)
assert train_campaigns.isdisjoint(test_campaigns)
assert validation_campaigns.isdisjoint(test_campaigns)

assert train_data["END_DAY"].max() < validation_data["START_DAY"].min()
assert validation_data["END_DAY"].max() < test_data["START_DAY"].min()

print("Split checks passed.")
print(f"Campaigns — train: {len(train_campaigns)}, "
      f"validation: {len(validation_campaigns)}, "
      f"test: {len(test_campaigns)}")

Split checks passed.
Campaigns — train: 13, validation: 4, test: 7


In [6]:
training_label_counts = train_data["redeemed"].value_counts().sort_index()

print("Training label counts:")
print(training_label_counts)

print(
    f"\nTraining redemption rate: "
    f"{train_data['redeemed'].mean():.2%}"
)

Training label counts:
redeemed
0    2234
1     265
Name: count, dtype: int64

Training redemption rate: 10.60%


In [7]:
feature_columns = [
    "baskets_90d",
    "retailer_sales_90d",
    "recency_days_90d",
    "has_transactions_90d",
    "avg_retailer_sales_per_basket_90d",
    "active_days_90d"
]

X_train = train_data[feature_columns].copy()
y_train = train_data["redeemed"].copy()

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data["redeemed"].copy()

print(f"Training input shape: {X_train.shape}")
print(f"Validation input shape: {X_validation.shape}")

Training input shape: (2499, 6)
Validation input shape: (1584, 6)


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

print(baseline_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])


In [9]:
baseline_pipeline.fit(X_train, y_train)

print("Logistic regression baseline trained.")
print(
    "Optimization iterations:",
    baseline_pipeline.named_steps["model"].n_iter_[0]
)

Logistic regression baseline trained.
Optimization iterations: 13


In [10]:
from sklearn.metrics import roc_auc_score, average_precision_score

validation_probabilities = baseline_pipeline.predict_proba(
    X_validation
)[:, 1]

print(f"Validation redemption rate: {y_validation.mean():.2%}")
print(
    f"Validation ROC-AUC: "
    f"{roc_auc_score(y_validation, validation_probabilities):.4f}"
)
print(
    f"Validation average precision: "
    f"{average_precision_score(y_validation, validation_probabilities):.4f}"
)

Validation redemption rate: 14.08%
Validation ROC-AUC: 0.6662
Validation average precision: 0.2145


In [11]:
import numpy as np
from sklearn.metrics import brier_score_loss

constant_probabilities = np.full(
    len(y_validation),
    y_train.mean()
)

print(
    f"Logistic regression Brier score: "
    f"{brier_score_loss(y_validation, validation_probabilities):.4f}"
)
print(
    f"Constant baseline Brier score: "
    f"{brier_score_loss(y_validation, constant_probabilities):.4f}"
)

Logistic regression Brier score: 0.1186
Constant baseline Brier score: 0.1222


In [12]:
coefficient_table = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": baseline_pipeline.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)

coefficient_table.round(3)

,feature,coefficient
3,has_transactions_90d,0.275
1,retailer_sales_90d,0.231
5,active_days_90d,0.180
4,avg_retailer_sales_per_basket_90d,0.156
0,baskets_90d,-0.166
2,recency_days_90d,-0.317


In [13]:
validation_review = validation_data[
    ["CAMPAIGN", "DESCRIPTION", "redeemed"]
].copy()

validation_review["predicted_probability"] = validation_probabilities

validation_campaign_summary = (
    validation_review.groupby(["CAMPAIGN", "DESCRIPTION"])
    .agg(
        examples=("redeemed", "size"),
        observed_redemption_rate=("redeemed", "mean"),
        mean_predicted_probability=("predicted_probability", "mean")
    )
)

validation_campaign_summary.round(4)

,,examples,observed_redemption_rate,mean_predicted_probability
CAMPAIGN,DESCRIPTION,,,
10,TypeB,123,0.0813,0.1237
11,TypeB,214,0.0280,0.1200
12,TypeB,170,0.0647,0.1227
13,TypeA,1077,0.1820,0.1049


In [14]:
candidate_feature_columns = feature_columns + ["DESCRIPTION"]

X_train_with_type = train_data[candidate_feature_columns].copy()
X_validation_with_type = validation_data[candidate_feature_columns].copy()

print(f"Training input shape: {X_train_with_type.shape}")
print(f"Validation input shape: {X_validation_with_type.shape}")
print(
    "Campaign types in training:",
    sorted(X_train_with_type["DESCRIPTION"].unique())
)

Training input shape: (2499, 7)
Validation input shape: (1584, 7)
Campaign types in training: ['TypeA', 'TypeB', 'TypeC']


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

candidate_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        feature_columns
    ),
    (
        "campaign_type",
        OneHotEncoder(handle_unknown="ignore"),
        ["DESCRIPTION"]
    )
])

campaign_type_pipeline = Pipeline([
    ("preprocessor", candidate_preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

print("Campaign-type pipeline defined; not trained yet.")

Campaign-type pipeline defined; not trained yet.


In [16]:
campaign_type_pipeline.fit(X_train_with_type, y_train)

print("Campaign-type logistic regression trained.")
print(
    "Optimization iterations:",
    campaign_type_pipeline.named_steps["model"].n_iter_[0]
)

Campaign-type logistic regression trained.
Optimization iterations: 31


In [18]:
type_validation_probabilities = campaign_type_pipeline.predict_proba(
    X_validation_with_type
)[:, 1]

validation_comparison = pd.DataFrame([
    {
        "model": name,
        "roc_auc": roc_auc_score(y_validation, probabilities),
        "average_precision": average_precision_score(
            y_validation, probabilities
        ),
        "brier_score": brier_score_loss(y_validation, probabilities)
    }
    for name, probabilities in [
        ("Numerical baseline", validation_probabilities),
        ("With campaign type", type_validation_probabilities)
    ]
])

validation_comparison.round(4)

,model,roc_auc,average_precision,brier_score
0,Numerical baseline,0.6662,0.2145,0.1186
1,With campaign type,0.7409,0.2832,0.1138


In [19]:
validation_review["type_predicted_probability"] = (
    type_validation_probabilities
)

campaign_comparison = (
    validation_review.groupby(["CAMPAIGN", "DESCRIPTION"])
    .agg(
        examples=("redeemed", "size"),
        observed_rate=("redeemed", "mean"),
        numerical_prediction=("predicted_probability", "mean"),
        with_type_prediction=("type_predicted_probability", "mean")
    )
)

campaign_comparison.round(4)

,,examples,observed_rate,numerical_prediction,with_type_prediction
CAMPAIGN,DESCRIPTION,,,,
10,TypeB,123,0.0813,0.1237,0.0623
11,TypeB,214,0.0280,0.1200,0.0599
12,TypeB,170,0.0647,0.1227,0.0616
13,TypeA,1077,0.1820,0.1049,0.1316


In [20]:
evaluation_folder = project_root / "reports" / "model_evaluation"
evaluation_folder.mkdir(parents=True, exist_ok=True)

validation_comparison.to_csv(
    evaluation_folder / "logistic_regression_validation.csv",
    index=False
)

campaign_comparison.reset_index().to_csv(
    evaluation_folder / "logistic_regression_by_campaign.csv",
    index=False
)

print(f"Saved two validation reports to: {evaluation_folder}")

Saved two validation reports to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/reports/model_evaluation


In [21]:
import joblib

model_folder = project_root / "models"
model_folder.mkdir(parents=True, exist_ok=True)

joblib.dump(
    baseline_pipeline,
    model_folder / "logistic_numeric_baseline.joblib"
)

joblib.dump(
    campaign_type_pipeline,
    model_folder / "logistic_with_campaign_type.joblib"
)

print(f"Saved both trained pipelines to: {model_folder}")

Saved both trained pipelines to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/models


In [22]:
import json

modeling_config = {
    "target": "redeemed",
    "numeric_features": feature_columns,
    "campaign_type_features": candidate_feature_columns,
    "history_window_days": 90,
    "eligible_campaign_end_day_cutoff": 704,
    "cutoff_note": "Conservative modeling choice, not a confirmed collection end date.",
    "validation_start_day": validation_start_day,
    "test_start_day": test_start_day,
    "exclude_boundary_crossing_campaigns": True
}

config_path = model_folder / "modeling_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(modeling_config, file, indent=2)

print(f"Saved modeling configuration to: {config_path}")

Saved modeling configuration to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/models/modeling_config.json
